# 1. 목적과 범위

주거실태조사 2016–2024년 원자료로 다음 두 지표를 시도별로 산출한다.

- 청년가구 자가점유비율: 조사연도-가구주 출생연도가 20–39인 가구 중 점유형태 1–7을 분모, 자가(1)를 분자로 한 가중 비율
- 청년가구 월소득 대비 주택임대료 비율(RIR): 청년 임차가구(점유형태 2–6)의 가구별 연간 환산 주거비를 연소득으로 나눈 비율의 가중평균

연령은 조사일 기준 만 나이가 아니라 자료 제약상 `조사연도 - 출생연도`로 계산한다. 주택가격 지표, 원자료 통합본, 공식 HCC CSV는 만들지 않는다.

## 2. 청년가구 기준과 산식

청년가구는 `조사연도 - 가구주 출생연도`가 20–39인 가구다. 이는 조사 기준일을 반영한 정확한 만나이가 아니라 연도 기준 연령이다. 2016년은 직접 가구주 출생연도를, 2017–2024년은 관계코드 1인 가구원이 정확히 한 명일 때 같은 번호의 출생연도를 사용한다.

자가점유비율은 유효 점유형태 1–7 청년가구의 가중치 합 중 자가(1)의 가중치 비율이다. RIR은 임차(2–6) 청년가구별 `HCC = 보증금 × 전환율 / 100 + 월세 × 12`, `RIR = HCC / (월가구소득 × 12) × 100`을 계산한 뒤 가구 가중치로 평균한다.

## 3. 라이브러리와 원자료 경로 설정

저장소 루트를 기준으로 원자료와 `data/processed` 경로를 설정한다.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data").is_dir() else cwd.parent
RAW_DIR = REPO_ROOT / "data" / "raw" / "구조환경지수 원데이터 구축용" / "주거실태조사"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
YEARS = list(range(2016, 2025))
INVALID_CODE = 9_999_999

assert RAW_DIR.is_dir(), RAW_DIR
assert PROCESSED_DIR.is_dir(), PROCESSED_DIR
print(f"저장소: {REPO_ROOT}")
print(f"원자료: {RAW_DIR}")

저장소: D:\University\yumocha\yumocha-issue49
원자료: D:\University\yumocha\yumocha-issue49\data\raw\구조환경지수 원데이터 구축용\주거실태조사


## 4. 주거실태조사 9개 CSV 목록 생성

각 연도에 정확히 하나의 재추출 CSV가 대응하는지 확인하고, CP949 헤더의 개수·중복·공란 및 예상 변수 구성을 검증한다. 원자료는 수정하지 않는다.

In [2]:
COL_TENURE = "문7. 귀 댁의 점유형태는 어디에 해당됩니까?"
COL_DEPOSIT = "문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_보증금(만원)"
COL_RENT = "문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_월세(만원)"
COL_INCOME = "문49. 지난 1년간의 월평균 가구 소득_6) 월평균 총 경상소득(만원)"
COL_BIRTH_2016 = "문2-1. 귀 댁의 가구주는 몇 년도에 태어나셨습니까?"
REL_COLS = [f"가구 구성원 No.{number}_가구주와의 관계" for number in range(1, 12)]
BIRTH_COLS = [f"가구 구성원 No.{number}_출생연도" for number in range(1, 12)]
REL_ALIASES = [f"가구주와의관계_No{number}" for number in range(1, 12)]
BIRTH_ALIASES = [f"출생연도_No{number}" for number in range(1, 12)]

housing_files = {}
header_rows = []
for year in YEARS:
    matches = sorted(RAW_DIR.glob(f"{year}_일반가구_*.csv"))
    assert len(matches) == 1, f"{year}년 파일 후보 {len(matches)}개: {matches}"
    path = matches[0]
    columns = pd.read_csv(path, encoding="cp949", nrows=0).columns.tolist()
    assert len(columns) == len(set(columns)), f"{year}년 중복 헤더"
    assert all(str(column).strip() for column in columns), f"{year}년 빈 헤더"
    weight_column = "모집단가중치" if year == 2016 else ("최종 가중치" if year == 2020 else "최종가중치")
    expected = (
        ["시도", COL_BIRTH_2016, COL_TENURE, COL_DEPOSIT, COL_RENT, COL_INCOME, weight_column]
        if year == 2016
        else ["시도", COL_TENURE, COL_DEPOSIT, COL_RENT, *REL_COLS, *BIRTH_COLS, COL_INCOME, weight_column]
    )
    assert columns == expected, f"{year}년 예상 헤더와 불일치"
    housing_files[year] = path
    header_rows.append({"연도": year, "파일명": path.name, "열 수": len(columns), "가중치 헤더": weight_column})

header_check = pd.DataFrame(header_rows)
print(header_check.to_string(index=False))

  연도                          파일명  열 수 가중치 헤더
2016 2016_일반가구_20260727_16717.csv    7 모집단가중치
2017 2017_일반가구_20260727_16717.csv   28  최종가중치
2018 2018_일반가구_20260727_16717.csv   28  최종가중치
2019 2019_일반가구_20260727_16717.csv   28  최종가중치
2020 2020_일반가구_20260727_16717.csv   28 최종 가중치
2021 2021_일반가구_20260727_94806.csv   28  최종가중치
2022 2022_일반가구_20260727_94806.csv   28  최종가중치
2023 2023_일반가구_20260727_94806.csv   28  최종가중치
2024 2024_일반가구_20260727_94806.csv   28  최종가중치


## 5. KOSIS 전월세전환율 전처리와 연평균 계산

KOSIS CSV의 `종합`·`지역별 전월세전환율`만 선택한다. 월별 값 `5.8`은 5.8%이므로 HCC 계산에서 `/100`을 정확히 한 번 적용한다. 내보낸 파일의 `단위` 열이 공란이면 이를 기록하되, 과업에서 확정한 퍼센트 의미를 적용한다.

In [3]:
kosis_matches = sorted(RAW_DIR.glob("*KOSIS*전월세전환율*.csv"))
assert len(kosis_matches) == 1, f"KOSIS 파일 후보 {len(kosis_matches)}개: {kosis_matches}"
KOSIS_PATH = kosis_matches[0]
kosis_raw = pd.read_csv(KOSIS_PATH, encoding="cp949")
trailing_columns = [column for column in kosis_raw.columns if str(column).startswith("Unnamed:")]
assert all(kosis_raw[column].isna().all() for column in trailing_columns), "비어 있지 않은 Unnamed 열 존재"
kosis = kosis_raw.drop(columns=trailing_columns)
id_columns = ["주택유형별", "지역별", "항목", "단위"]
assert all(column in kosis.columns for column in id_columns)
month_pattern = re.compile(r"^(20(?:16|17|18|19|20|21|22|23|24))\.(0[1-9]|1[0-2]) 월$")
month_columns = [column for column in kosis.columns if month_pattern.fullmatch(str(column))]
assert len(month_columns) == 9 * 12, len(month_columns)
unit_values = {str(value).strip() for value in kosis["단위"].dropna() if str(value).strip()}
assert not unit_values or unit_values == {"%"}, f"예상 밖 단위: {unit_values}"
unit_note = "KOSIS 단위 열은 공란; 과업 정의에 따라 값을 퍼센트로 해석" if not unit_values else "KOSIS 단위: %"
print(f"KOSIS 파일: {KOSIS_PATH.name}")
print(unit_note)

KOSIS 파일: 16-24 KOSIS 지역별 전월세전환율.csv
KOSIS 단위 열은 공란; 과업 정의에 따라 값을 퍼센트로 해석


### 지역명 정규화와 연평균 전환율

전국과 17개 시도의 2016–2024년 각 연도에 월별 관측치가 정확히 12개인지 먼저 검증한 뒤 단순평균한다. `강원특별자치도`, `전북특별자치도` 표기는 각각 `강원`, `전북`으로 정규화한다.

In [4]:
REGION_ORDER = ["서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]
REGION_NORMALIZE = {"강원특별자치도": "강원", "전북특별자치도": "전북", "제주특별자치도": "제주"}
target_regions = ["전국", *REGION_ORDER]
selected = kosis.loc[(kosis["주택유형별"] == "종합") & (kosis["항목"] == "지역별 전월세전환율")].copy()
selected["지역"] = selected["지역별"].replace(REGION_NORMALIZE)
selected = selected.loc[selected["지역"].isin(target_regions)]
assert selected["지역"].value_counts().eq(1).all(), "대상 지역 중복"
assert set(selected["지역"]) == set(target_regions), "전국 또는 17개 시도 누락"
conversion_long = selected.melt(id_vars=["지역"], value_vars=month_columns, var_name="기준연월", value_name="전월세전환율")
date_parts = conversion_long["기준연월"].str.extract(month_pattern)
conversion_long["연도"] = date_parts[0].astype(int)
conversion_long["월"] = date_parts[1].astype(int)
conversion_long["전월세전환율"] = pd.to_numeric(conversion_long["전월세전환율"], errors="coerce")
month_counts = conversion_long.groupby(["지역", "연도"], observed=True)["월"].nunique()
nonmissing_counts = conversion_long.groupby(["지역", "연도"], observed=True)["전월세전환율"].count()
assert month_counts.eq(12).all() and nonmissing_counts.eq(12).all(), "지역-연도별 12개월 완전성 실패"
annual_conversion = conversion_long.groupby(["지역", "연도"], observed=True)["전월세전환율"].mean()
assert annual_conversion.size == 18 * 9
print(f"KOSIS 12개월 완전성: {int(month_counts.eq(12).sum())}/{month_counts.size} 지역-연도 통과")
print(annual_conversion.unstack("연도").round(3).to_string())

KOSIS 12개월 완전성: 162/162 지역-연도 통과
연도   2016   2017   2018   2019   2020   2021   2022   2023   2024
지역                                                               
강원  8.185  7.618  7.270  7.060  6.746  6.837  6.626  6.919  6.919
경기  6.628  6.393  6.372  6.266  5.953  5.901  6.088  6.408  6.291
경남  8.063  7.661  7.291  7.157  6.881  6.990  6.855  6.673  6.468
경북  9.890  9.486  9.079  8.861  8.515  8.404  7.884  7.440  7.632
광주  7.515  7.084  6.956  6.856  6.461  6.273  6.197  6.022  5.896
대구  7.759  7.421  7.326  7.314  7.119  6.830  6.469  6.207  6.070
대전  7.489  7.333  7.117  6.891  6.556  6.106  5.983  6.275  6.522
부산  7.323  7.039  6.907  6.541  6.316  5.982  5.919  6.013  6.094
서울  5.888  5.497  5.347  5.189  4.929  4.740  4.812  5.166  5.113
세종  5.684  5.158  5.456  5.339  5.111  5.065  5.528  6.138  5.965
울산  7.614  7.391  7.201  7.155  6.809  6.652  6.621  6.822  6.746
인천  7.094  6.821  6.661  6.453  6.035  5.874  5.925  6.548  6.307
전국  6.720  6.382  6.254  6.120  5.814  5.67

## 6. 연도별 가구주 연령 판별

2016년은 가구주 출생연도 직접 변수를 사용한다. 2017–2024년은 관계코드가 1인 구성원이 정확히 한 명인 경우에만 대응 출생연도를 사용하며, No.1을 가구주로 가정하지 않는다.

원자료에서 전세(2)의 월세와 무보증 월세 계열(4–6)의 보증금은 구조적 공란이다. 이 조합에만 0을 부여한다. 실제 0은 유지하며, 명시적 결측 코드 `9999999`와 그 밖의 공란은 제외한다. 가중치는 양수인 경우만 사용한다.

In [ ]:
SURVEY_REGION_MAP = {11: "서울", 21: "부산", 22: "대구", 23: "인천", 24: "광주", 25: "대전", 26: "울산", 29: "세종", 31: "경기", 32: "강원", 33: "충북", 34: "충남", 35: "전북", 36: "전남", 37: "경북", 38: "경남", 39: "제주"}

def weighted_mean(values, weights, mask):
    valid = mask & values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return float(np.average(values.loc[valid], weights=weights.loc[valid]))

def regional_weighted_mean(frame, value_column, mask):
    return pd.Series(
        {region: weighted_mean(frame[value_column], frame["가구가중치"], mask & frame["지역"].eq(region)) for region in REGION_ORDER},
        dtype=float,
    )

def load_and_prepare_year(year, path):
    weight_source = "모집단가중치" if year == 2016 else ("최종 가중치" if year == 2020 else "최종가중치")
    common_sources = ["시도", COL_TENURE, COL_DEPOSIT, COL_RENT, COL_INCOME, weight_source]
    usecols = [*common_sources, COL_BIRTH_2016] if year == 2016 else [*common_sources, *REL_COLS, *BIRTH_COLS]
    raw = pd.read_csv(path, encoding="cp949", usecols=usecols)
    aliases = {"시도": "시도코드", COL_TENURE: "점유형태", COL_DEPOSIT: "보증금_원자료", COL_RENT: "월세_원자료", COL_INCOME: "월가구소득", weight_source: "가구가중치", **dict(zip(REL_COLS, REL_ALIASES)), **dict(zip(BIRTH_COLS, BIRTH_ALIASES))}
    frame = raw.rename(columns=aliases)

    relation_excluded = 0
    if year == 2016:
        head_birth = pd.to_numeric(frame[COL_BIRTH_2016], errors="coerce").mask(lambda series: series.eq(INVALID_CODE))
    else:
        relations = frame[REL_ALIASES].apply(pd.to_numeric, errors="coerce")
        births = frame[BIRTH_ALIASES].apply(pd.to_numeric, errors="coerce").mask(lambda data: data.eq(INVALID_CODE))
        head_flags = relations.eq(1)
        head_count = head_flags.sum(axis=1)
        relation_excluded = int(head_count.ne(1).sum())
        head_birth = births.where(head_flags.to_numpy()).max(axis=1).where(head_count.eq(1))

    frame["가구주출생연도"] = head_birth
    frame["연령"] = year - frame["가구주출생연도"]
    frame["청년가구"] = frame["연령"].between(20, 39, inclusive="both")
    frame["시도코드"] = pd.to_numeric(frame["시도코드"], errors="coerce")
    frame["지역"] = frame["시도코드"].map(SURVEY_REGION_MAP)
    if year == 2016:
        frame.loc[frame["시도코드"].isin([29, 34]), "지역"] = "충남"
    assert frame["지역"].notna().all(), f"{year}년 미매핑 시도코드 존재"
    frame["점유형태"] = pd.to_numeric(frame["점유형태"], errors="coerce")
    frame["가구가중치"] = pd.to_numeric(frame["가구가중치"], errors="coerce")
    frame["월가구소득"] = pd.to_numeric(frame["월가구소득"], errors="coerce").mask(lambda series: series.eq(INVALID_CODE))

    deposit_raw = pd.to_numeric(frame["보증금_원자료"], errors="coerce")
    rent_raw = pd.to_numeric(frame["월세_원자료"], errors="coerce")
    frame["보증금"] = deposit_raw.mask(deposit_raw.eq(INVALID_CODE))
    frame["월세"] = rent_raw.mask(rent_raw.eq(INVALID_CODE))
    structural_zero_deposit = frame["점유형태"].isin([4, 5, 6]) & deposit_raw.isna()
    structural_zero_rent = frame["점유형태"].eq(2) & rent_raw.isna()
    frame.loc[structural_zero_deposit, "보증금"] = 0
    frame.loc[structural_zero_rent, "월세"] = 0

    rate_map = annual_conversion.xs(year, level="연도").to_dict()
    frame["전월세전환율"] = frame["지역"].map(rate_map)
    frame["연간환산주거비"] = frame["보증금"] * frame["전월세전환율"] / 100 + frame["월세"] * 12
    frame["RIR"] = frame["연간환산주거비"] / (frame["월가구소득"] * 12) * 100
    return frame, relation_excluded

def calculate_year(year, path):
    frame, relation_excluded = load_and_prepare_year(year, path)
    positive_weight = frame["가구가중치"].notna() & frame["가구가중치"].gt(0)
    valid_tenure = frame["점유형태"].isin(range(1, 8))
    renter = frame["점유형태"].isin(range(2, 7))
    young = frame["청년가구"]
    valid_housing_cost = frame["보증금"].notna() & frame["월세"].notna() & frame["보증금"].ge(0) & frame["월세"].ge(0)
    valid_income = frame["월가구소득"].notna() & frame["월가구소득"].gt(0)
    valid_conversion = frame["전월세전환율"].notna()
    young_renter = young & renter
    valid_rir = young_renter & valid_housing_cost & valid_income & valid_conversion & positive_weight
    assert not (young_renter & ~positive_weight).any(), f"{year}년 청년 임차가구 가중치 결측/비양수"

    own_eligible = young & valid_tenure & positive_weight
    own_indicator = regional_weighted_mean(frame.assign(자가=frame["점유형태"].eq(1).astype(float) * 100), "자가", own_eligible)
    rir_indicator = regional_weighted_mean(frame, "RIR", valid_rir)
    if year == 2016:
        own_indicator.loc["세종"] = np.nan
        rir_indicator.loc["세종"] = np.nan

    qa = {
        "연도": year,
        "관계코드1 없음/복수 제외": relation_excluded,
        "청년 임차가구": int(young_renter.sum()),
        "유효 RIR 가구": int(valid_rir.sum()),
        "소득·주거비 결측/비정상 제외": int((young_renter & ~(valid_housing_cost & valid_income)).sum()),
        "전환율 매칭 누락": int((young_renter & ~valid_conversion).sum()),
        "_RIR100초과": int((valid_rir & frame["RIR"].gt(100)).sum()),
    }

    if year == 2024:
        all_households = valid_tenure & positive_weight
        all_renter = renter & valid_housing_cost & valid_conversion & positive_weight
        current_own_national = weighted_mean(frame["점유형태"].eq(1).astype(float) * 100, frame["가구가중치"], own_eligible)
        national_rate = float(annual_conversion.loc[("전국", year)])
        national_hcc = frame["보증금"] * national_rate / 100 + frame["월세"] * 12
        aux_rows = []
        for region in ["제주", "전국"]:
            region_mask = frame["지역"].eq(region) if region != "전국" else pd.Series(True, index=frame.index)
            hcc_values = frame["연간환산주거비"] if region != "전국" else national_hcc
            aux_rows.append({
                "지역": region,
                "공식산식대응_전체가구_자가점유비율": weighted_mean(frame["점유형태"].eq(1).astype(float) * 100, frame["가구가중치"], all_households & region_mask),
                "현재지표_청년가구_자가점유비율": current_own_national if region == "전국" else own_indicator.loc[region],
                "공식산식대응_전체임차가구_HCC": weighted_mean(hcc_values, frame["가구가중치"], all_renter & region_mask),
                "보조값_청년임차가구_HCC": weighted_mean(hcc_values, frame["가구가중치"], young_renter & valid_housing_cost & valid_conversion & positive_weight & region_mask),
                "현재지표_청년임차가구_RIR": weighted_mean(frame["RIR"], frame["가구가중치"], valid_rir & region_mask),
            })
        auxiliary = pd.DataFrame(aux_rows)
    else:
        auxiliary = None

    return own_indicator, rir_indicator, qa, auxiliary


### 연도별 처리와 QA

메모리 사용을 줄이고 연도별 구조 차이를 분리하기 위해 9개 원자료를 한 번에 합치지 않고 연도별로 읽고 계산한다. 관계코드 1 없음/복수, 청년 임차가구, 유효 RIR, 제외 및 전환율 매칭 누락 건수를 출력한다.

In [6]:
own_by_year = {}
rir_by_year = {}
qa_rows = []
auxiliary_2024 = None
for year in YEARS:
    own_by_year[year], rir_by_year[year], qa, auxiliary = calculate_year(year, housing_files[year])
    qa_rows.append(qa)
    if auxiliary is not None:
        auxiliary_2024 = auxiliary

qa_table = pd.DataFrame(qa_rows)
large_rir_total = int(qa_table.pop("_RIR100초과").sum())
print(qa_table.to_string(index=False))
if large_rir_total:
    print(f"주의: RIR 100% 초과 가구 {large_rir_total:,}건을 발견했으며 합의된 기준이 없어 제거하거나 조정하지 않음")

  연도  관계코드1 없음/복수 제외  청년 임차가구  유효 RIR 가구  소득·주거비 결측/비정상 제외  전환율 매칭 누락
2016               0     1874       1844                30          0
2017               0     5412       5328                84          0
2018               0     6032       5878               154          0
2019               0     6350       6179               171          0
2020               0     5403       5168               235          0
2021               0     5072       4926               146          0
2022               0     4957       4788               169          0
2023               0     6274       6119               155          0
2024               0     6486       6338               148          0
주의: RIR 100% 초과 가구 250건을 발견했으며 합의된 기준이 없어 제거하거나 조정하지 않음


## 7. 청년가구 자가점유비율

분모는 청년가구 중 점유형태 1–7의 가중치 합, 분자는 그중 자가(1)의 가중치 합이다. 결과는 퍼센트 단위다.

In [7]:
own_output = pd.DataFrame({"지역": REGION_ORDER, "세부지표": "청년가구 자가점유비율"})
for year in YEARS:
    own_output[str(year)] = own_by_year[year].reindex(REGION_ORDER).to_numpy()
own_output.loc[:, [str(year) for year in YEARS]] = own_output[[str(year) for year in YEARS]].round(1)
jeju_own_2024 = float(own_output.loc[own_output["지역"].eq("제주"), "2024"].iloc[0])
print(f"2024년 제주 청년가구 자가점유비율: {jeju_own_2024:.1f}%")
print(own_output.to_string(index=False))

2024년 제주 청년가구 자가점유비율: 25.8%
지역        세부지표  2016  2017  2018  2019  2020  2021  2022  2023  2024
서울 청년가구 자가점유비율  19.0  18.4  18.0  16.5  14.8  14.4  12.2  13.3  10.2
부산 청년가구 자가점유비율  44.4  38.0  35.7  40.1  32.3  27.2  24.5  27.8  24.1
대구 청년가구 자가점유비율  53.2  36.3  38.1  36.3  30.5  27.9  28.0  27.2  31.2
인천 청년가구 자가점유비율  39.8  36.4  39.6  37.6  37.5  30.5  37.0  29.6  27.1
광주 청년가구 자가점유비율  36.3  38.8  30.7  36.4  33.4  26.4  30.3  31.1  25.6
대전 청년가구 자가점유비율  27.3  27.8  24.3  24.9  21.5  16.9  17.3  16.2  18.4
울산 청년가구 자가점유비율  59.4  47.9  47.5  47.6  44.0  38.6  36.4  33.6  37.2
세종 청년가구 자가점유비율   NaN  27.2  34.0  32.0  33.9  34.7  30.9  31.9  29.2
경기 청년가구 자가점유비율  33.6  33.5  32.3  29.5  27.2  22.9  23.1  26.2  22.6
강원 청년가구 자가점유비율  26.8  29.2  25.7  31.6  31.8  31.4  27.8  21.8  23.5
충북 청년가구 자가점유비율  36.3  37.4  34.2  36.7  35.4  26.9  28.6  26.6  31.1
충남 청년가구 자가점유비율  38.0  40.2  41.6  40.1  36.3  29.3  32.9  30.5  28.2
전북 청년가구 자가점유비율  37.5  35.8  38.1  39.5  36.2  29.0  29.8  26.5  26.2
전남 청년가

## 8. 청년가구 월소득 대비 주택임대료 비율

임차가구별 `HCC = 보증금 × 지역·연도 연평균 전월세전환율 / 100 + 월세 × 12`, `RIR = HCC / (월가구소득 × 12) × 100`을 계산한 뒤 RIR을 최종가중치로 가중평균한다. 소득 0 이하·결측, 명시적 임차료 결측, 음수, 전환율 누락 및 비양수 가중치는 제외하며 극단값은 제거하지 않는다.

In [8]:
rir_output = pd.DataFrame({"지역": REGION_ORDER, "세부지표": "청년가구 월소득 대비 주택임대료 비율"})
for year in YEARS:
    rir_output[str(year)] = rir_by_year[year].reindex(REGION_ORDER).to_numpy()
rir_output.loc[:, [str(year) for year in YEARS]] = rir_output[[str(year) for year in YEARS]].round(1)
jeju_rir_2024 = float(rir_output.loc[rir_output["지역"].eq("제주"), "2024"].iloc[0])
print(f"2024년 제주 청년가구 월소득 대비 주택임대료 비율: {jeju_rir_2024:.1f}%")
print(rir_output.to_string(index=False))

2024년 제주 청년가구 월소득 대비 주택임대료 비율: 16.5%
지역                 세부지표  2016  2017  2018  2019  2020  2021  2022  2023  2024
서울 청년가구 월소득 대비 주택임대료 비율  28.9  30.1  28.9  28.1  28.7  27.9  26.4  26.7  26.3
부산 청년가구 월소득 대비 주택임대료 비율  28.0  25.1  23.9  25.2  24.3  23.4  26.7  29.3  21.5
대구 청년가구 월소득 대비 주택임대료 비율  20.2  25.8  24.0  25.4  25.7  21.9  27.1  21.1  25.4
인천 청년가구 월소득 대비 주택임대료 비율  20.1  21.4  21.8  20.0  22.6  21.5  22.6  22.0  19.8
광주 청년가구 월소득 대비 주택임대료 비율  20.2  22.3  21.6  21.4  21.8  20.6  22.8  19.0  21.6
대전 청년가구 월소득 대비 주택임대료 비율  35.4  23.0  26.5  25.2  24.6  25.0  22.9  25.3  20.0
울산 청년가구 월소득 대비 주택임대료 비율  20.1  21.2  25.8  21.2  21.8  19.8  20.7  20.6  19.4
세종 청년가구 월소득 대비 주택임대료 비율   NaN  23.6  23.1  24.0  20.0  23.4  23.2  26.5  25.6
경기 청년가구 월소득 대비 주택임대료 비율  25.2  23.0  23.5  24.3  22.1  20.4  22.3  23.0  20.4
강원 청년가구 월소득 대비 주택임대료 비율  25.9  28.4  27.5  26.1  20.0  21.2  22.3  20.6  29.5
충북 청년가구 월소득 대비 주택임대료 비율  22.4  26.8  20.4  24.5  19.1  22.0  26.6  22.5  26.1
충남 청년가구 월소득 대비 주택임대료 비율  30

## 9. 연도별 결과를 최종 데이터프레임으로 변환

연도별 시도 결과를 요청한 지역·연도 순서의 두 최종 데이터프레임으로 변환했다. 과업에서 확정한 2016년 조사구조 규칙에 따라 시도코드 29와 34를 충남·세종 결합표본으로 묶어 충남 결과를 계산하고, 두 표본 모두 충남 연평균 전월세전환율을 적용한다. 세종은 두 결과 모두 결측으로 두며 충남 값을 세종에 복사하지 않는다.

In [9]:
own_sejong_2016 = own_output.loc[own_output["지역"].eq("세종"), "2016"].iloc[0]
rir_sejong_2016 = rir_output.loc[rir_output["지역"].eq("세종"), "2016"].iloc[0]
assert pd.isna(own_sejong_2016) and pd.isna(rir_sejong_2016)
print("2016년 세종: 자가점유비율·RIR 모두 결측 유지 확인")

2016년 세종: 자가점유비율·RIR 모두 결측 유지 확인


## 10. 제주·전국 및 공식 원지표 대응 보조값 확인

제주여성가족연구원 원지표는 전체 가구 자가점유비율과 전체 임차가구의 연간 환산 주거비(HCC)다. 현재 결과는 청년가구 필터를 적용한 자가점유비율과 청년 임차가구의 가구별 RIR 가중평균이므로 서로 다른 지표다. 아래 2024년 제주·전국 보조값은 방법론 비교용이며 CSV로 저장하지 않는다. 저장소와 제공 자료에서 직접 대조할 공식 발표 수치는 확인되지 않아 수치 일치 검증은 수행할 수 없다.

In [10]:
assert auxiliary_2024 is not None
print(auxiliary_2024.round(1).to_string(index=False))
print("공식 발표 비교값: 저장소·제공 자료에서 미확인 — 직접 일치 검증 불가")

지역  공식원지표_전체가구_자가점유비율  현재지표_청년가구_자가점유비율  공식원지표_전체임차가구_HCC  보조값_청년임차가구_HCC  현재지표_청년임차가구_RIR
제주               58.5              25.8             648.4           643.9             16.5
전국               58.4              22.4             880.9           817.5             22.9
공식 발표 비교값: 저장소·제공 자료에서 미확인 — 직접 일치 검증 불가


## 11. 최종 CSV 저장

각 파일을 `지역 | 세부지표 | 2016 | … | 2024` 구조의 17행으로 만들고, 값은 소수점 첫째 자리까지 반올림해 UTF-8-SIG로 저장한다. 전국 행과 추가 지표는 저장하지 않는다.

In [11]:
OWN_PATH = PROCESSED_DIR / "청년가구_자가점유비율_2016-2024.csv"
RIR_PATH = PROCESSED_DIR / "청년가구_월소득_대비_주택임대료_비율_2016-2024.csv"
own_output.to_csv(OWN_PATH, index=False, encoding="utf-8-sig", float_format="%.1f")
rir_output.to_csv(RIR_PATH, index=False, encoding="utf-8-sig", float_format="%.1f")
print(f"저장: {OWN_PATH} / shape={own_output.shape}")
print(f"저장: {RIR_PATH} / shape={rir_output.shape}")

저장: D:\University\yumocha\yumocha-issue49\data\processed\청년가구_자가점유비율_2016-2024.csv / shape=(17, 11)
저장: D:\University\yumocha\yumocha-issue49\data\processed\청년가구_월소득_대비_주택임대료_비율_2016-2024.csv / shape=(17, 11)


## 12. 재읽기 검증과 핵심 결과

저장된 CSV를 UTF-8-SIG로 다시 읽어 열 순서, 지역 순서, 17×11 크기, 숫자형·소수점 한 자리, 2016년 세종 결측 및 2024년 제주 값을 검증한다.

In [12]:
expected_columns = ["지역", "세부지표", *[str(year) for year in YEARS]]
reloaded = {
    "자가점유비율": pd.read_csv(OWN_PATH, encoding="utf-8-sig"),
    "RIR": pd.read_csv(RIR_PATH, encoding="utf-8-sig"),
}
for label, data in reloaded.items():
    assert data.shape == (17, 11), (label, data.shape)
    assert data.columns.tolist() == expected_columns, (label, data.columns.tolist())
    assert data["지역"].tolist() == REGION_ORDER, f"{label} 지역 순서 불일치"
    numeric = data[[str(year) for year in YEARS]].apply(pd.to_numeric, errors="coerce")
    finite = numeric.stack().dropna()
    assert np.allclose(finite * 10, np.round(finite * 10)), f"{label} 소수점 한 자리 위반"
    assert pd.isna(data.loc[data["지역"].eq("세종"), "2016"].iloc[0])

assert float(reloaded["자가점유비율"].loc[reloaded["자가점유비율"]["지역"].eq("제주"), "2024"].iloc[0]) == jeju_own_2024
assert float(reloaded["RIR"].loc[reloaded["RIR"]["지역"].eq("제주"), "2024"].iloc[0]) == jeju_rir_2024
print("재읽기 검증: 두 CSV 모두 17×11, 열·지역 순서, 숫자형/소수점 한 자리, 2016 세종 결측 통과")
print(f"2024 제주 — 자가점유비율 {jeju_own_2024:.1f}%, RIR {jeju_rir_2024:.1f}%")
print(f"KOSIS 완전성 — 18개 지역 × 9개 연도 = {month_counts.size}개 지역-연도, 각 12개월 통과")

재읽기 검증: 두 CSV 모두 17×11, 열·지역 순서, 숫자형/소수점 한 자리, 2016 세종 결측 통과
2024 제주 — 자가점유비율 25.8%, RIR 16.5%
KOSIS 완전성 — 18개 지역 × 9개 연도 = 162개 지역-연도, 각 12개월 통과
